In [ ]:
# To install relevant packages
!pip install spotipy
!pip install pandas
!pip install matplotlib
!pip install networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.5/261.5 kB 7.8 MB/s eta 0:00:00


In [ ]:
import spotipy
import spotipy.util as util
from spotipy.oauth2 import SpotifyClientCredentials
import spotipy.oauth2 as oauth2
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import time

client_id = '829be16fb27f43fd98c2332fcd335f7e' # Client ID (Arav)
client_secret = '309db9acfc034e08bb93d646aa13a120' #Client Secret (Arav)
market = ['US']
redirect_uri='https://google.com/'
scope = "user-library-read user-follow-read user-top-read playlist-read-private playlist-read-collaborative"

sp = spotipy.Spotify(
        auth_manager=spotipy.SpotifyOAuth(
          client_id=client_id,
          client_secret=client_secret,
          redirect_uri=redirect_uri,
          scope=scope, open_browser=False))

artist_graph = nx.Graph()

In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import networkx as nx

# Define credentials
client_id = '829be16fb27f43fd98c2332fcd335f7e'
client_secret = '309db9acfc034e08bb93d646aa13a120'
redirect_uri = 'http://127.0.0.1:8000/callback'
scope = "user-library-read user-follow-read user-top-read playlist-read-private playlist-read-collaborative"

# Auth manager with correct parameters
auth_manager = SpotifyOAuth(client_id=client_id,
                            client_secret=client_secret,
                            redirect_uri=redirect_uri,
                            scope=scope)

# Get the token
token_info = auth_manager.get_cached_token()
if not token_info:
    auth_manager.get_access_token()
    token_info = auth_manager.get_cached_token()

# Check if the token is valid
if token_info and not auth_manager.is_token_expired(token_info):
    access_token = token_info['access_token']
    sp = spotipy.Spotify(auth=access_token)
    print("Access token is valid")
else:
    print("Error: Failed to get valid access token.")
    exit()

# Create your graph
artist_graph = nx.Graph()
user_profile = sp.current_user()
print(f"Logged in as: {user_profile['display_name']}")


<ipython-input-3-340b79698061>:20: DeprecationWarning: You're using 'as_dict = True'.get_access_token will return the token string directly in future versions. Please adjust your code accordingly, or use get_cached_token instead.
  auth_manager.get_access_token()


In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import ipywidgets as widgets
from IPython.display import display
from urllib.parse import urlparse, parse_qs

# Define credentials
client_id = '829be16fb27f43fd98c2332fcd335f7e'
client_secret = '309db9acfc034e08bb93d646aa13a120'
redirect_uri = 'https://google.com/'  # Dummy URI (use any valid URI for manual authorization)
scope = "user-library-read user-follow-read user-top-read playlist-read-private playlist-read-collaborative"

# Auth manager with SpotifyOAuth but NO redirect on localhost
auth_manager = SpotifyOAuth(
    client_id=client_id,
    client_secret=client_secret,
    redirect_uri=redirect_uri,
    scope=scope,
    open_browser=False
)

# Get the authorization URL
auth_url = auth_manager.get_authorize_url()

# Create a clickable link in Colab
link_button = widgets.HTML(
    f"<a href='{auth_url}' target='_blank'><button>Click here to authorize Spotify</button></a>"
)
display(link_button)

# Get the redirect URL manually
redirect_response = input("After authorizing, paste the URL you were redirected to here: ")

# parsed_url = urlparse(redirect_response)
# authorization_code = parse_qs(parsed_url.query)['code'][0]  # Get the 'code' parameter

# Get the token after authorization
token_info = auth_manager.get_access_token(redirect_response)

# Validate the token
if token_info:
    sp = spotipy.Spotify(auth=token_info['access_token'])
    user_profile = sp.current_user()
    print(f"✅ Successfully logged in as: {user_profile['display_name']}")
else:
    print("❌ Error: Failed to get a valid token.")

def get_top_artists():
    load = {}
    results = sp.playlist_tracks(playlist_id='37i9dQZEVXbLRQDuF5jeBp', limit=50, market='US')
    artist_set = set()
    for item in results['items']:
        artist_data = item['track']['artists']
        for artist in artist_data:
            artist_set.add((artist['id'], artist['name']))
    return list(artist_set)

print(get_top_artists())

In [ ]:
import os
if os.path.exists(".cache"):
    os.remove(".cache")

In [ ]:
# Starting point: US Top 50 Track
#https://open.spotify.com/playlist/3ocM8VMdHSjut7hb7KYGAE?si=b941f385f13a41af
#https://open.spotify.com/playlist/37i9dQZF1DXcBWIGoYBM5M?si=50db394a8f004de1
#37i9dQZEVXbLRQDuF5jeBp
#37i9dQZEVXbMDoHDwVN2tF
#37i9dQZF1DXcBWIGoYBM5M
def get_top_artists():
    results = sp.playlist_tracks(playlist_id='37i9dQZEVXbLRQDuF5jeBp', limit=50, market='US')
    artist_set = set()
    for item in results['items']:
        artist_data = item['track']['artists']
        for artist in artist_data:
            artist_set.add((artist['id'], artist['name']))
    return list(artist_set)

In [ ]:
print(get_top_artists())

In [ ]:
results = sp.playlist(playlist_id='37i9dQZEVXbLRQDuF5jeBp', market='US')
print(results)

In [ ]:
import subprocess
import json

# Your access token from the Spotify OAuth process
access_token = 'YOUR_ACCESS_TOKEN'  # Replace with actual access token

# Playlist ID for Today's Top Hits
playlist_id = '37i9dQZF1DXcBWIGoYBM5M'

# `curl` command to get the playlist tracks
curl_command = [
    'curl', '-X', 'GET',
    f'https://api.spotify.com/v1/playlists/{playlist_id}/tracks',
    '-H', f'Authorization: Bearer {access_token}'
]

# Run the curl command and capture the response
response = subprocess.check_output(curl_command)

# Parse the response as JSON
data = json.loads(response)

# Extract artist names from the tracks
artist_set = set()
for item in data['items']:
    artist_data = item['track']['artists']
    for artist in artist_data:
        artist_set.add((artist['id'], artist['name']))

# Print out the list of artists
print(list(artist_set))


In [ ]:
taylor_uri = 'spotify:artist:06HL4z0CvFAxyc27GXpf02'

In [ ]:
artist_name = []
track_name = []
popularity = []
track_id = []
for i in range(0,950,50):
    track_results = sp.search(q='year:2018', type='track', limit=50,offset=i)
    for i, t in enumerate(track_results['tracks']['items']):
        artist_name.append(t['artists'][0]['name'])
        track_name.append(t['name'])
        track_id.append(t['id'])
        popularity.append(t['popularity'])

In [ ]:
track_dataframe = pd.DataFrame({'artist_name' : artist_name, 'track_name' : track_name, 'track_id' : track_id, 'popularity' : popularity})
print(track_dataframe.shape)
track_dataframe.head()

In [ ]:
artist_name = 'Taylor Swift'
results = sp.search(q=f"artist:{artist_name}", type="artist", limit=50, offset=0)
artist = results['artists']['items'][0]

print(artist)
# Print some info
print("Name:", artist['name'])
print("Genres:", artist['genres'])
print("Followers:", artist['followers']['total'])
print("Popularity:", artist['popularity'])
print("Spotify URL:", artist['external_urls']['spotify'])

{'external_urls': {'spotify': 'https://open.spotify.com/artist/06HL4z0CvFAxyc27GXpf02'}, 'followers': {'href': None, 'total': 134695555}, 'genres': [], 'href': 'https://api.spotify.com/v1/artists/06HL4z0CvFAxyc27GXpf02', 'id': '06HL4z0CvFAxyc27GXpf02', 'images': [{'url': 'https://i.scdn.co/image/ab6761610000e5ebe672b5f553298dcdccb0e676', 'height': 640, 'width': 640}, {'url': 'https://i.scdn.co/image/ab67616100005174e672b5f553298dcdccb0e676', 'height': 320, 'width': 320}, {'url': 'https://i.scdn.co/image/ab6761610000f178e672b5f553298dcdccb0e676', 'height': 160, 'width': 160}], 'name': 'Taylor Swift', 'popularity': 97, 'type': 'artist', 'uri': 'spotify:artist:06HL4z0CvFAxyc27GXpf02'}
Name: Taylor Swift
Genres: []
Followers: 134695555
Popularity: 97
Spotify URL: https://open.spotify.com/artist/06HL4z0CvFAxyc27GXpf02


In [ ]:
# Degree Centrality (how many connections an artist has)
degree_centrality = nx.degree_centrality(artist_graph)

# Betweenness Centrality (how often an artist serves as a bridge in the network)
betweenness_centrality = nx.betweenness_centrality(artist_graph)

# Closeness Centrality (how easily an artist can reach all other artists)
closeness_centrality = nx.closeness_centrality(artist_graph)

# PageRank (probabilistic measure of centrality based on connectivity)
pagerank = nx.pagerank(artist_graph)

# Display the top 10 artists by degree centrality
top_10_degree = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:10]
print("Top 10 artists by degree centrality:", top_10_degree)

# Display the top 10 artists by betweenness centrality
top_10_betweenness = sorted(betweenness_centrality.items(), key=lambda x: x[1], reverse=True)[:10]
print("Top 10 artists by betweenness centrality:", top_10_betweenness)

# Combine all centralities into a single dictionary (for convenience)
centralities = {
    'degree': degree_centrality,
    'betweenness': betweenness_centrality,
    'closeness': closeness_centrality,
    'pagerank': pagerank
}

In [ ]:
# Define a threshold for each centrality measure
degree_threshold = 0.1  # Can be adjusted based on your graph's size
betweenness_threshold = 0.01  # Can be adjusted based on the range of values
closeness_threshold = 0.4  # Can be adjusted based on the range of values
pagerank_threshold = 0.1  # Can be adjusted based on the range of values

# Filter artists whose centrality values are below the threshold
emerging_artists = []

for artist in artist_graph.nodes():
    if (degree_centrality[artist] < degree_threshold and
        betweenness_centrality[artist] < betweenness_threshold and
        closeness_centrality[artist] < closeness_threshold and
        pagerank[artist] < pagerank_threshold):
        emerging_artists.append(artist)

print("Emerging artists:", emerging_artists)